# Proyecto dataset de Titanic

Este proyecto tiene por objetivo analizar datos sobre los registros de sobrevivientes del titanic. Utilizando como base de información los archivos que se encuentran en https://www.kaggle.com/competitions/titanic/data. El fin es poder predecir mediante sus datos cuales son las personas que sobreviven o no. Además de identificar cuales son los factores que mas influyen en las probabilidades de sobrevivir.

Algunas explicaciones de columnas:  
- pclass = Clase del ticket: 1st, 2nd, 3rd
- sibsp = Número de hermanos o cónyugues arriba del titanic
- parch = Número de familiares o hijos arriba del titanic
- ticket = Número del ticket
- fare = Tarifa
- cabin = Número de la cabina
- embarked = Puerto de embarcación: C(Cherbourg), Q(Queenstown), S(Southampton)

### Imports necesarios

In [ ]:
import pandas as pd
import random
import plotly.express as px
from sklearn.feature_selection

### Carga de datos

In [2]:
ruta_train_csv = r'../data/raw/train.csv'
df_inicial_train = pd.read_csv(ruta_train_csv)

### Visualización inicial

DataFrame cargado inicial:

In [3]:
print(f'Datos iniciales:\n{df_inicial_train}')

Datos iniciales:
     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 .

Info del DataFrame y describe

In [4]:
df_inicial_train.info()
print(f'\nDescribe:\n{df_inicial_train.describe()}')

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB

Describe:
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.486592    0.83

### EDA

Rellenar valores de Age con la mediana

In [5]:
info_rellenar_datos_age = {'Age': df_inicial_train['Age'].median()}
df_edad_rellenada = df_inicial_train.fillna(info_rellenar_datos_age)
print(f'DataFrame con Age completo:\n{df_edad_rellenada}')
df_edad_rellenada.info()

DataFrame con Age completo:
     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                       

Quitar la columna Cabin, para este ejercicio no se considerará debido a su cantidad de valores distintos, no se podría clasificar

In [6]:
df_drop_cabin = df_edad_rellenada.drop(columns=['Cabin'])

Rellenar los valores faltantes de Embarked, son pocos pero esto es para tener un DataFrame mas limpio y fácil de trabajar. Se aplica la funcion lambda en donde es "na" se elige un Embarked aleatorio

In [7]:
valores_posibles_embarked = df_drop_cabin['Embarked'].dropna().unique()
print(valores_posibles_embarked)
serie_embarked_procesado = df_drop_cabin['Embarked'].apply(
    lambda x: random.choice(valores_posibles_embarked) if pd.isna(x) else x
)
df_procesado = df_drop_cabin
df_procesado['Embarked'] = serie_embarked_procesado

df_procesado.info()

<StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Embarked     891 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


### Creación de gráficos para entendimiento

Crear columna de sobrevivientes en palabras, esto para crear un gráfico mas entendible y amigable

In [8]:
df_procesado['Sobreviviente'] = df_procesado['Survived'].map({0: 'No sobrevivió', 1: 'Sobrevivió'})
print(f'DataFrame con columna de sobreviviente o no:\n{df_procesado}')

DataFrame con columna de sobreviviente o no:
     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                      

1. Generar DataFrame solo de mujeres, para contar cuantos sobrevivientes y no sobrevivientes hay. Luego mapear colores para que en el gráfico se vean colores especificos dependiendo si es sobreviviente o no.

In [9]:
df_mujeres = df_procesado[df_procesado['Sex'] == 'female']
cuenta_sobrevivientes_mujeres = df_mujeres['Sobreviviente'].value_counts()
cuenta_sobrevivientes_mujeres = cuenta_sobrevivientes_mujeres.reset_index()
print(cuenta_sobrevivientes_mujeres)
mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}



   Sobreviviente  count
0     Sobrevivió    233
1  No sobrevivió     81


1.1 Gráfico de sobrevivientes mujeres

In [10]:
fig_pie_sobrevivientes_female = px.pie(
    cuenta_sobrevivientes_mujeres,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_female.show()

2. Realizar misma accion para los hombres

In [11]:
df_hombres = df_procesado[df_procesado['Sex'] == 'male']
cuenta_sobrevivientes_hombres = df_hombres['Sobreviviente'].value_counts()
cuenta_sobrevivientes_hombres = cuenta_sobrevivientes_hombres.reset_index()
print(cuenta_sobrevivientes_hombres)
mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}


   Sobreviviente  count
0  No sobrevivió    468
1     Sobrevivió    109


2.2 Gráfico de sobrevivientes hombres

In [12]:
fig_pie_sobrevivientes_hombres = px.pie(
    cuenta_sobrevivientes_hombres,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_hombres.show()

3. Ahora buscar taza de sobrevivientes por rangos de edad, primero utilizar una cantidad considerable (mayor a 100) registros por cada grupo para identificar si hay alguna tendencia y que sea fidedigna

In [13]:
df_jovenes_debajo_18 = df_procesado[df_procesado['Age'] <= 18]
df_adultos_18_40 = df_procesado[(df_procesado['Age'] > 18) & (df_procesado['Age'] <= 40)]
df_adultos_sobre_40 = df_procesado[df_procesado['Age'] > 40]
print(len(df_jovenes_debajo_18))
print(len(df_adultos_18_40))
print(len(df_adultos_sobre_40))

139
602
150


3.1 Crear y manejar datos para ver sobrevivientes jovenes

In [14]:
sobrevivientes_debajo_18 = df_jovenes_debajo_18['Sobreviviente'].value_counts()
sobrevivientes_debajo_18 = sobrevivientes_debajo_18.reset_index()
print(sobrevivientes_debajo_18)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}


   Sobreviviente  count
0     Sobrevivió     70
1  No sobrevivió     69


3.2 Gráfico de sobrevivientes "jovenes"

In [15]:
fig_pie_sobrevivientes_jovenes = px.pie(
    sobrevivientes_debajo_18,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)
fig_pie_sobrevivientes_jovenes.show()

3.3 Realizar lo mismo para los pasajeros con edad entre 18 y 40 años

In [16]:
sobrevivientes_18_40 = df_adultos_18_40['Sobreviviente'].value_counts()
sobrevivientes_18_40 = sobrevivientes_18_40.reset_index()
print(sobrevivientes_18_40)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}


   Sobreviviente  count
0  No sobrevivió    385
1     Sobrevivió    217


3.4 Gráfico sobrevivientes entre 18 y 40 años

In [17]:
fig_pie_sobrevivientes_18_40 = px.pie(
    sobrevivientes_18_40,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_18_40.show()

3.5 Realizar lo mismo para pasajeros mayores a 40 años

In [18]:
sobrevivientes_mayores_40 = df_adultos_sobre_40['Sobreviviente'].value_counts()
sobrevivientes_mayores_40 = sobrevivientes_mayores_40.reset_index()
print(sobrevivientes_mayores_40)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}



   Sobreviviente  count
0  No sobrevivió     95
1     Sobrevivió     55


3.6 Gráfico para sobrevivientes mayores a 40 años

In [19]:
fig_pie_sobrevivientes_mayores_40 = px.pie(
    sobrevivientes_mayores_40,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_mayores_40.show()

4. Para este apartado analizar si el precio de la tarifa (Fare) puede tener que ver con la sobrevivencia, pero antes saber que se puede conocer como Fare alto o bajo, para ello realizar un histograma

In [20]:
fig_histograma_fares = px.histogram(
    df_procesado,
    x='Fare'
)
fig_histograma_fares.show()

4.1 Ahora conociendo los datos de que casi todos los pasajeros tenian un fare de 15 o menor, se separa por ese número

In [21]:
df_fare_bajo = df_procesado[df_procesado['Fare'] <= 15]
cuenta_sobrevivientes_fare_bajo = df_fare_bajo['Sobreviviente'].value_counts()
cuenta_sobrevivientes_fare_bajo = cuenta_sobrevivientes_fare_bajo.reset_index()
print(cuenta_sobrevivientes_fare_bajo)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}



   Sobreviviente  count
0  No sobrevivió    344
1     Sobrevivió    114


4.2 Gráfico para sobrevivientes de "Fare" bajo

In [22]:
fig_sobrevivientes_fare_bajo = px.pie(
    cuenta_sobrevivientes_fare_bajo,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_sobrevivientes_fare_bajo.show()

4.3 Crear DataFrame con registros con "Fare" mas alto

In [23]:
df_fare_alto = df_procesado[df_procesado['Fare'] > 15]
cuenta_sobrevivientes_fare_alto = df_fare_alto['Sobreviviente'].value_counts()
cuenta_sobrevivientes_fare_alto = cuenta_sobrevivientes_fare_alto.reset_index()
print(cuenta_sobrevivientes_fare_alto)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}


   Sobreviviente  count
0     Sobrevivió    228
1  No sobrevivió    205


4.4 Gráfico para pasajeros con "Fare" mas alto

In [24]:
fig_sobrevivientes_fare_alto = px.pie(
    cuenta_sobrevivientes_fare_alto,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_sobrevivientes_fare_alto.show()

5. Generar DataFrame con la información necesaria para ver si la "Pclass" tiene que ver con la sobrevivencia

5.1 Primero obtener datos de "Pclass" = 1

In [ ]:
df_pclass_1 = df_procesado[df_procesado['Pclass'] == 1]
sobrevivientes_pclass_1 = df_pclass_1['Sobreviviente'].value_counts()
sobrevivientes_pclass_1 = sobrevivientes_pclass_1.reset_index()
print(sobrevivientes_pclass_1)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

   Sobreviviente  count
0     Sobrevivió    136
1  No sobrevivió     80


5.2 Gráfico de sobrevivientes en Pclass = 1

In [27]:
fig_pie_sobrevivientes_pclass_1 = px.pie(
    sobrevivientes_pclass_1,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_pclass_1.show()

5.3 Datos para Pclass = 2

In [28]:
df_pclass_2 = df_procesado[df_procesado['Pclass'] == 2]
sobrevivientes_pclass_2 = df_pclass_2['Sobreviviente'].value_counts()
sobrevivientes_pclass_2 = sobrevivientes_pclass_2.reset_index()
print(sobrevivientes_pclass_2)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

   Sobreviviente  count
0  No sobrevivió     97
1     Sobrevivió     87


5.4 Gráfico sobrevivientes en Pclass = 2

In [29]:
fig_pie_sobrevivientes_pclass_2 = px.pie(
    sobrevivientes_pclass_2,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_pclass_2.show()

5.5 Datos para Pclass = 3

In [30]:
df_pclass_3 = df_procesado[df_procesado['Pclass'] == 3]
sobrevivientes_pclass_3 = df_pclass_3['Sobreviviente'].value_counts()
sobrevivientes_pclass_3 = sobrevivientes_pclass_3.reset_index()
print(sobrevivientes_pclass_3)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

   Sobreviviente  count
0  No sobrevivió    372
1     Sobrevivió    119


5.6 Gráfico sobrevivientes en Pclass = 3

In [31]:
fig_pie_sobrevivientes_pclass_3 = px.pie(
    sobrevivientes_pclass_3,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_pclass_3.show()

6. Obtener información para saber si el lugar donde fue embarcado tiene relación

6.1 Info de Embarked = C

In [34]:
df_embarked_c = df_procesado[df_procesado['Embarked'] == 'C']
sobrevivientes_embarked_c = df_embarked_c['Sobreviviente'].value_counts()
sobrevivientes_embarked_c = sobrevivientes_embarked_c.reset_index()
print(sobrevivientes_embarked_c)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

   Sobreviviente  count
0     Sobrevivió     93
1  No sobrevivió     75


6.2 Gráfico sobrevivientes Embarked = C

In [38]:
fig_pie_sobrevivientes_embarked_c = px.pie(
    sobrevivientes_embarked_c,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_embarked_c.show()

6.4 Info Embarked = Q

In [37]:
df_embarked_q = df_procesado[df_procesado['Embarked'] == 'Q']
sobrevivientes_embarked_q = df_embarked_q['Sobreviviente'].value_counts()
sobrevivientes_embarked_q = sobrevivientes_embarked_q.reset_index()
print(sobrevivientes_embarked_q)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

   Sobreviviente  count
0  No sobrevivió     47
1     Sobrevivió     31


6.5 Gráfico sobrevivientes Embarked = Q

In [39]:
fig_pie_sobrevivientes_embarked_q = px.pie(
    sobrevivientes_embarked_q,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_embarked_q.show()

6.6 Info Embarked = S

In [36]:
df_embarked_s = df_procesado[df_procesado['Embarked'] == 'S']
sobrevivientes_embarked_s = df_embarked_s['Sobreviviente'].value_counts()
sobrevivientes_embarked_s = sobrevivientes_embarked_s.reset_index()
print(sobrevivientes_embarked_s)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

   Sobreviviente  count
0  No sobrevivió    427
1     Sobrevivió    218


6.7 Gráfico sobrevivientes Embarked = S

In [40]:
fig_pie_sobrevivientes_embarked_s = px.pie(
    sobrevivientes_embarked_s,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_embarked_s.show()

### Machine Learning

Para el trabajo con machine learning la idea es poder aprender con los datos que tenemos y después poder predecir información

1. Primero quitaremos columnas que no vamos a utilizar, cambiar los valores de Sex a 0 y 1. Además lógica para crear 3 columnas, una para cada tipo de Embarked, marcando 1 en donde corresponda.

In [ ]:
df_ml = df_procesado.drop(columns=['PassengerId', 'Name', 'Sibsp', 'Parch', 'Ticket', 'Sobreviviente'])

df_ml['Sex'] = df_ml['Sex'].map({'female': 0, 'male': 1})

df_ml['Embarked_C'] = (df_ml['Embarked'] == 'C').astype(int)
df_ml['Embarked_Q'] = (df_ml['Embarked'] == 'Q').astype(int)
df_ml['Embarked_S'] = (df_ml['Embarked'] == 'S').astype(int)

df_ml = df_ml.drop(columns=['Embarked'])

2. Separar los datos para entrenamiento y pruebas

In [ ]:
X = df_ml.drop(columns='Survived')
y = df_ml['Survived']